In [ ]:
import scipy.io as sio
import scipy.signal as signal
import numpy as np
import torch
import os

class CSIPreprocessorBase:
    """Filtrage + resample + normalisation, sans augmentation"""
    def __init__(self, fs=200.0, cutoff=10.0, filter_order=3):
        self.fs = fs
        self.cutoff = cutoff
        self.filter_order = filter_order
        nyq = 0.5 * fs
        normal_cutoff = cutoff / nyq
        self.b, self.a = signal.butter(filter_order, normal_cutoff, btype='low')

    def load_mat(self, filepath):
        mat = sio.loadmat(filepath)
        return mat["CSIamp"].astype(np.float32)

    def filter_signal(self, data):
        return signal.filtfilt(self.b, self.a, data, axis=1)

    def resample(self, data):
        return signal.resample(data, 500, axis=1)

    def normalize(self, data):
        mean = np.mean(data, axis=1, keepdims=True)
        std = np.std(data, axis=1, keepdims=True)
        std = np.where(std < 1e-8, 1e-8, std)
        return (data - mean) / std

    def to_tensor(self, data):
        data = np.expand_dims(data, axis=0)
        return torch.from_numpy(data).float()

    def __call__(self, filepath):
        data = self.load_mat(filepath)
        data = self.filter_signal(data)
        data = self.resample(data)
        data = self.normalize(data)
        return self.to_tensor(data)

def collect_all_data(data_dir, classes, label_map, preprocessor):
    X, y = [], []
    for class_name in classes:
        class_path = os.path.join(data_dir, class_name)
        if not os.path.exists(class_path):
            continue
        for f in os.listdir(class_path):
            if f.endswith('.mat'):
                tensor = preprocessor(os.path.join(class_path, f))
                X.append(tensor)
                y.append(label_map[class_name])
    return torch.stack(X), torch.tensor(y, dtype=torch.long)

# Configuration
DATASET_PATH = "NTU-Fi_HAR"
TRAIN_DIR = os.path.join(DATASET_PATH, "train_amp")
TEST_DIR  = os.path.join(DATASET_PATH, "test_amp")
CLASSES = ["box","circle","clean","fall","run","walk"]
label_map = {c:i for i,c in enumerate(CLASSES)}

prep = CSIPreprocessorBase()

# Prétraiter TOUTES les données d'entraînement (sans split)
X_all_train, y_all_train = collect_all_data(TRAIN_DIR, CLASSES, label_map, prep)

# Prétraiter le vrai test (séparé physiquement)
X_test, y_test = collect_all_data(TEST_DIR, CLASSES, label_map, prep)

print(f"Toutes les données d'entraînement : {X_all_train.shape}")
print(f"Test : {X_test.shape}")

# Sauvegarder les tenseurs prétraités (non splittés)
os.makedirs("processed_data", exist_ok=True)
torch.save((X_all_train, y_all_train), "processed_data/preprocessed_all_train.pt")
torch.save((X_test, y_test), "processed_data/preprocessed_test.pt")
print("Sauvegarde terminée : données prétraitées (split à faire ensuite).")

Train: torch.Size([748, 1, 342, 500]), Val: torch.Size([188, 1, 342, 500]), Test: torch.Size([264, 1, 342, 500])
Sauvegarde terminée : données prétraitées .


In [1]:
print("Prétraitement terminé : données prétraitées sans augmentation.")

Prétraitement terminé : données prétraitées sans augmentation.
